<a href="https://colab.research.google.com/github/docling-project/docling/blob/main/docs/examples/backend_xml_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conversion of custom XML

| Step | Tech | Execution | 
| --- | --- | --- |
| Embedding | Hugging Face / Sentence Transformers | 💻 Local |
| Vector store | Milvus | 💻 Local |
| Gen AI | Hugging Face Inference API | 🌐 Remote | 

## Overview

This is an example of using [Docling](https://docling-project.github.io/docling/) for converting structured data (XML) into a unified document
representation format, `DoclingDocument`, and leverage its enriched structured content for RAG applications.

Data used in this example consist of open-access medical and scientific articles in [JATS](https://jats.nlm.nih.gov/) (Journal Article Tag Suite) XML, the format used by [PubMed Central® (PMC)](https://pmc.ncbi.nlm.nih.gov/) and many scholarly publishers.

In this notebook, we accomplish the following:
- [Simple conversion](#simple-conversion) of supported XML files in a nutshell
- An [end-to-end application](#end-to-end-application) using public collections of XML files supported by Docling
  - [Setup](#setup) the API access for generative AI
  - [Fetch the data](#fetch-the-data) from a collection of JATS articles, using Docling custom backends
  - [Parse, chunk, and index](#parse-chunk-and-index) the documents in a vector database
  - [Perform RAG](#question-answering-with-rag) using [LlamaIndex Docling extension](../../integrations/llamaindex/)

For more details on document chunking with Docling, refer to the [Chunking](../../concepts/chunking/) documentation. For RAG with Docling and LlamaIndex, also check the example [RAG with LlamaIndex](../rag_llamaindex/).

## Simple conversion

The XML file format defines and stores data in a format that is both human-readable and machine-readable.
Because of this flexibility, Docling requires custom backend processors to interpret XML definitions and convert them into `DoclingDocument` objects.

Some public data collections in XML format are already supported by Docling (USTPO patents and PMC articles). In these cases, the document conversion is straightforward and the same as with any other supported format, such as PDF or HTML. The execution example in [Simple Conversion](../minimal/) is the recommended usage of Docling for a single file:

In [68]:
from docling.document_converter import DocumentConverter

# a sample PMC article:
source = "../../tests/data/jats/sources/elife-56337.nxml"
converter = DocumentConverter()
result = converter.convert(source)
print(result.status)

2026-09-23 13:26:07,659 - INFO - detected formats: [<InputFormat.XML_JATS: 'xml_jats'>, <InputFormat.XML_XBRL: 'xml_xbrl'>, <InputFormat.XML_DOCLANG: 'xml_doclang'>, <InputFormat.XML_USPTO: 'xml_uspto'>]
2026-09-23 13:26:07,661 - INFO - Going to convert document batch...
2026-09-23 13:26:07,661 - INFO - Initializing pipeline for SimplePipeline with options hash c51d76e87a563ae9551305f3279088c5
2026-09-23 13:26:07,662 - INFO - Processing document elife-56337.nxml
2026-09-23 13:26:07,706 - INFO - Finished converting document elife-56337.nxml in 0.05 sec.


ConversionStatus.SUCCESS


Once the document is converted, it can be exported to any format supported by Docling. For instance, to markdown (showing here the first lines only):

In [69]:
md_doc = result.document.export_to_markdown()

delim = "\n"
print(delim.join(md_doc.split(delim)[:8]))

# KRAB-zinc finger protein gene expansion in response to active retrotransposons in the murine lineage

Gernot Wolf, Alberto de Iaco, Ming-An Sun, Melania Bruno, Matthew Tinkham, Don Hoang, Apratim Mitra, Sherry Ralls, Didier Trono, Todd S Macfarlan

The Eunice Kennedy Shriver National Institute of Child Health and Human Development, The National Institutes of Health, Bethesda, United States; School of Life Sciences, École Polytechnique Fédérale de Lausanne (EPFL), Lausanne, Switzerland

## Abstract



If the XML file is not supported, a `ConversionError` message will be raised.

In [70]:
from io import BytesIO

from docling.datamodel.base_models import DocumentStream
from docling.exceptions import ConversionError

xml_content = (
    b'<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE docling_test SYSTEM '
    b'"test.dtd"><docling>Random content</docling>'
)
stream = DocumentStream(name="docling_test.xml", stream=BytesIO(xml_content))
try:
    result = converter.convert(stream)
except ConversionError as ce:
    print(ce)

2026-09-23 13:26:40,933 - INFO - detected formats: [<InputFormat.XML_JATS: 'xml_jats'>, <InputFormat.XML_XBRL: 'xml_xbrl'>, <InputFormat.XML_DOCLANG: 'xml_doclang'>, <InputFormat.XML_USPTO: 'xml_uspto'>]
2026-09-23 13:26:40,935 - ERROR - Input document docling_test.xml with format None does not match any allowed format: (dict_keys([<InputFormat.DOCX: 'docx'>, <InputFormat.DOC: 'doc'>, <InputFormat.RTF: 'rtf'>, <InputFormat.PPTX: 'pptx'>, <InputFormat.PPT: 'ppt'>, <InputFormat.HTML: 'html'>, <InputFormat.MHTML: 'mhtml'>, <InputFormat.IMAGE: 'image'>, <InputFormat.PDF: 'pdf'>, <InputFormat.ASCIIDOC: 'asciidoc'>, <InputFormat.MD: 'md'>, <InputFormat.CSV: 'csv'>, <InputFormat.XLSX: 'xlsx'>, <InputFormat.XLS: 'xls'>, <InputFormat.ODT: 'odt'>, <InputFormat.ODS: 'ods'>, <InputFormat.ODP: 'odp'>, <InputFormat.XML_USPTO: 'xml_uspto'>, <InputFormat.XML_JATS: 'xml_jats'>, <InputFormat.XML_XBRL: 'xml_xbrl'>, <InputFormat.XML_DOCLANG: 'xml_doclang'>, <InputFormat.DCLX: 'dclx'>, <InputFormat.METS_GB

Conversion failed for: docling_test.xml with status: skipped. Errors: File format not allowed: docling_test.xml


You can always refer to the [Usage](../../usage/#supported-formats) documentation page for a list of supported formats.

## End-to-end application

This section describes a step-by-step application for processing XML files from supported public collections and use them for question-answering.

### Setup

Requirements can be installed as shown below. The `--no-warn-conflicts` argument is meant for Colab's pre-populated Python environment, feel free to remove for stricter usage.

In [39]:
%pip install -q --progress-bar off --no-warn-conflicts llama-index-core llama-index-readers-docling llama-index-node-parser-docling llama-index-embeddings-huggingface llama-index-llms-huggingface-api llama-index-vector-stores-milvus llama-index-readers-file python-dotenv

I0923 12:33:27.625738   34697 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


/home/neebal/Desktop/docling/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


Instead of running that command, you may run the following in your environment:<br>
```bash
uv add llama-index-core llama-index-readers-docling llama-index-node-parser-docling llama-index-embeddings-huggingface llama-index-llms-huggingface-api llama-index-vector-stores-milvus llama-index-readers-file python-dotenv
```

This notebook uses HuggingFace's Inference API. For an increased LLM quota, a token can be provided via the environment variable `HF_TOKEN`.

If you're running this notebook in Google Colab, make sure you [add](https://medium.com/@parthdasawant/how-to-use-secrets-in-google-colab-450c38e3ec75) your API key as a secret.

In [71]:
import os
from warnings import filterwarnings

from dotenv import load_dotenv


def _get_env_from_colab_or_os(key):
    try:
        from google.colab import userdata

        try:
            return userdata.get(key)
        except userdata.SecretNotFoundError:
            pass
    except ImportError:
        pass
    return os.getenv(key)


load_dotenv()

filterwarnings(action="ignore", category=UserWarning, module="pydantic")

We can now define the main parameters:

In [72]:
from pathlib import Path
from tempfile import mkdtemp

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
EMBED_MODEL = HuggingFaceEmbedding(model_name=EMBED_MODEL_ID)
TEMP_DIR = Path(mkdtemp())
MILVUS_URI = str(TEMP_DIR / "docling.db")
GEN_MODEL = HuggingFaceInferenceAPI(
    token=_get_env_from_colab_or_os("HF_TOKEN"),
    model_name="Qwen/Qwen2.5-72B-Instruct",
)
embed_dim = len(EMBED_MODEL.get_text_embedding("hi"))
# https://github.com/huggingface/transformers/issues/5486:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

2026-09-23 13:28:47,008 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-23 13:28:47,038 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json?%2FBAAI%2Fbge-small-en-v1.5%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-09-23 13:28:47,353 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-23 13:28:47,378 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json?%2FBAAI%2Fbge-small-en-v1.5%2Fresolve%2Fmain%2Fconfig_sentence_transformers.json=&etag=%22dcb0c0d97d09b930d13600b1a773ddb27e441aab%22 "HTTP/1.1 200 OK"
2026-

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-09-23 13:28:50,046 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-23 13:28:50,317 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-23 13:28:50,575 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-23 13:28:50,835 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-09-23 13:28:51,083 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-23 13:28:51,099 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json?

### Fetch the data

In this notebook we will use JATS XML articles, a collection format supported by Docling out of the box.

Docling ships a small set of JATS articles as test data, which we use here as a ready-made corpus so the example runs fully offline. The same `DoclingReader` shown below also works on JATS files fetched from the [PubMed Central® Open Access FTP server](https://ftp.ncbi.nlm.nih.gov/pub/pmc/), where each article is distributed as a `.tar.gz` archive containing the full-text XML.

#### JATS articles corpus

We assemble a small, topically diverse corpus from the JATS sources bundled with Docling's test data (under `tests/data/jats/sources/`). Each file is a full-text article that Docling parses with its JATS backend into a richly structured `DoclingDocument`.

In [73]:
from pathlib import Path

# Docling bundles a small collection of JATS (Journal Article Tag Suite) XML
# articles as test data. We reuse them here as a ready-made corpus so the
# example runs fully offline.
JATS_DIR = Path("../../tests/data/jats/sources")

# Background corpus: indexed via SimpleDirectoryReader further below.
BG_FILES = [
    JATS_DIR / "pmc2231364.nxml",    # Yersinia pestis transcriptional response
    JATS_DIR / "pntd.0008301.nxml",  # Lymphatic filariasis pre-TAS survey risk factors
    JATS_DIR / "pone.0234687.nxml",  # Greenhouse gas emissions in dairy cattle systems
    JATS_DIR / "ptag100.xml",        # 3D oscillator model of quark-antiquark systems
]

missing = [str(p) for p in BG_FILES if not p.exists()]
assert not missing, f"Missing JATS sources: {missing}"
print(f"Prepared {len(BG_FILES)} background JATS articles.")

Prepared 4 background JATS articles.


#### Focus article

We hold one additional JATS article aside as the *focus* document for the retrieval-augmented generation comparison below. Keeping it separate lets us attach explicit metadata to it and later filter the query engine so it answers strictly from this article's content.

In [74]:
FOCUS_ARTICLE = JATS_DIR / "elife-56337.nxml"  # KRAB-zinc finger genes vs. retrotransposons
assert FOCUS_ARTICLE.exists(), f"Missing focus article: {FOCUS_ARTICLE}"
print(f"Focus article: {FOCUS_ARTICLE.name}")

Focus article: elife-56337.nxml


In [75]:
print(f"Corpus ready: {len(BG_FILES)} background articles + 1 focus article ({FOCUS_ARTICLE.name}).")

Corpus ready: 4 background articles + 1 focus article (elife-56337.nxml).


### Parse, chunk, and index

The `DoclingDocument` format of the converted articles has a rich hierarchical structure, inherited from the original JATS XML and preserved by the Docling custom backend.
In this notebook, we will leverage:
- The `SimpleDirectoryReader` pattern to iterate over the JATS article files assembled in section [Fetch the data](#fetch-the-data).
- The LlamaIndex extensions, `DoclingReader` and `DoclingNodeParser`, to ingest the article chunks into a Milvus vector store.
- The `HierarchicalChunker` implementation, which applies a document-based hierarchical chunking, to leverage the article structures like sections and paragraphs within sections.

Refer to other possible implementations and usage patterns in the [Chunking](../../concepts/chunking/) documentation and the [RAG with LlamaIndex](../rag_llamaindex/) notebook.

##### Set the Docling reader and the directory reader

Note that `DoclingReader` uses Docling's `DocumentConverter` by default and therefore it will recognize the format of the XML files and leverage the JATS backend automatically.

We point the directory reader at the background JATS articles assembled in section [Fetch the data](#fetch-the-data).

In [76]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.docling import DoclingReader

reader = DoclingReader(export_type=DoclingReader.ExportType.JSON)
dir_reader = SimpleDirectoryReader(
    input_files=[str(p) for p in BG_FILES],
    file_extractor={".xml": reader, ".nxml": reader},
    filename_as_id=True,
)

##### Set the node parser

Note that the `HierarchicalChunker` is the default chunking implementation of the `DoclingNodeParser`.

In [77]:
from llama_index.node_parser.docling import DoclingNodeParser

node_parser = DoclingNodeParser()

##### Set a local Milvus database and run the ingestion

In [78]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.milvus import MilvusVectorStore

vector_store = MilvusVectorStore(
    uri=MILVUS_URI,
    dim=embed_dim,
    overwrite=True,
)

index = VectorStoreIndex.from_documents(
    documents=dir_reader.load_data(show_progress=True),
    transformations=[node_parser],
    storage_context=StorageContext.from_defaults(vector_store=vector_store),
    embed_model=EMBED_MODEL,
    show_progress=True,
)

2026-09-23 13:37:27,088 - INFO - MilvusLite server started for /tmp/tmpswy43q0o/docling.db on port 41175


2026-09-23 13:37:27,121 - ERROR - Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1168, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Loading files:   0%|          | 0/4 [00:00<?, ?it/s]

2026-09-23 13:37:27,654 - INFO - detected formats: [<InputFormat.XML_JATS: 'xml_jats'>, <InputFormat.XML_XBRL: 'xml_xbrl'>, <InputFormat.XML_DOCLANG: 'xml_doclang'>, <InputFormat.XML_USPTO: 'xml_uspto'>]
2026-09-23 13:37:27,657 - INFO - Going to convert document batch...
2026-09-23 13:37:27,658 - INFO - Initializing pipeline for SimplePipeline with options hash c51d76e87a563ae9551305f3279088c5
2026-09-23 13:37:27,659 - INFO - Processing document pmc2231364.nxml
2026-09-23 13:37:27,663 - INFO - Finished converting document pmc2231364.nxml in 0.01 sec.
2026-09-23 13:37:27,666 - INFO - detected formats: [<InputFormat.XML_JATS: 'xml_jats'>, <InputFormat.XML_XBRL: 'xml_xbrl'>, <InputFormat.XML_DOCLANG: 'xml_doclang'>, <InputFormat.XML_USPTO: 'xml_uspto'>]
2026-09-23 13:37:27,670 - INFO - Going to convert document batch...
2026-09-23 13:37:27,672 - INFO - Processing document pntd.0008301.nxml
2026-09-23 13:37:27,707 - INFO - Finished converting document pntd.0008301.nxml in 0.04 sec.
2026-09

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/254 [00:00<?, ?it/s]

Finally, add the focus article to the vector store directly from the reader, attaching a `file_name` to its metadata so we can filter on it later.

In [79]:
index.from_documents(
    documents=reader.load_data(
        FOCUS_ARTICLE, extra_info={"file_name": FOCUS_ARTICLE.name}
    ),
    transformations=[node_parser],
    storage_context=StorageContext.from_defaults(vector_store=vector_store),
    embed_model=EMBED_MODEL,
)

2026-09-23 13:38:54,751 - INFO - detected formats: [<InputFormat.XML_JATS: 'xml_jats'>, <InputFormat.XML_XBRL: 'xml_xbrl'>, <InputFormat.XML_DOCLANG: 'xml_doclang'>, <InputFormat.XML_USPTO: 'xml_uspto'>]
2026-09-23 13:38:54,765 - INFO - Going to convert document batch...
2026-09-23 13:38:54,768 - INFO - Processing document elife-56337.nxml
2026-09-23 13:38:54,831 - INFO - Finished converting document elife-56337.nxml in 0.09 sec.


### Question-answering with RAG

The retriever can be used to identify highly relevant documents:

In [80]:
retriever = index.as_retriever(similarity_top_k=3)
results = retriever.retrieve(
    "What risk factors are associated with failing pre-transmission assessment "
    "surveys (pre-TAS) in lymphatic filariasis elimination programs?"
)

for item in results:
    print(item)

Node ID: bc636293-a4d1-4783-9f59-9af852fae1e0
Text: This study aims to understand which factors are associated with
the need for additional rounds of MDA as identified by pre-TAS results
using programmatic data from 13 countries. The factors associated with
failing pre-TAS are not well understood and have not previously been
examined at a multi-country scale in the literature. We examine the
asso...
Score:  0.942

Node ID: 759055bd-d85f-4917-ae70-3c6c8c126e5c
Text: Building on previous work, we delineated five domains of
variables that could influence pre-TAS outcomes: prevalence, agent,
environment, MDA, and pre-TAS implementation (Table 1) [6–8]. We
prioritized key concepts that could be measured through our data or
captured through publicly available global geospatial data sets.
Score:  0.933

Node ID: 1704a636-e5af-4f24-a41f-8d347541420c
Text: This paper provides evidence from analysis of 554 districts and
13 countries on the factors associated with pre-TAS results. Baseline
preval

With the query engine, we can run the question-answering with the RAG pattern on the set of indexed documents.

First, we can prompt the LLM directly:

In [81]:
from llama_index.core.base.llms.types import ChatMessage, MessageRole
from rich.console import Console
from rich.panel import Panel

console = Console()
query = (
    "In this study, which retrotransposon families did the murine KRAB-ZFPs "
    "target, and what did the CRISPR/Cas9 deletion of the KRAB-ZFP gene "
    "clusters reveal?"
)

usr_msg = ChatMessage(role=MessageRole.USER, content=query)
response = GEN_MODEL.chat(messages=[usr_msg])

console.print(Panel(query, title="Prompt", border_style="bold red"))
console.print(
    Panel(
        response.message.content.strip(),
        title="Generated Content",
        border_style="bold green",
    )
)

2026-09-23 13:40:58,750 - INFO - HTTP Request: POST https://router.huggingface.co/v1/chat/completions "HTTP/1.1 200 OK"


╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│ In this study, which retrotransposon families did the murine KRAB-ZFPs target, and what did the CRISPR/Cas9     │
│ deletion of the KRAB-ZFP gene clusters reveal?                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Generated Content ───────────────────────────────────────────────╮
│ In the study you're referring to, murine KRAB-ZFPs (Krüppel-associated box domain-containing zinc finger        │
│ proteins) were found to target specific retrotransposon families. The primary retrotransposon families targeted │
│ by these KRAB-ZFPs include:                                                                                     │
│                                                                                                                 │
│ 1. **MusD (Murine D-type endogenous retrovirus)**: This is a type of endogenous retrovirus that is active in    │
│ the mouse genome.                                                                                               │
│ 2. **ETn (Early Transposon)**: These are another class of retrotransposons that are active in mice and are      │
│ known to cause insertional mutagenesis.                                                                         │
│                                                                                                                 │
│ ### CRISPR/Cas9 Deletion of KRAB-ZFP Gene Clusters                                                              │
│                                                                                                                 │
│ The CRISPR/Cas9-mediated deletion of KRAB-ZFP gene clusters revealed several important findings:                │
│                                                                                                                 │
│ 1. **Increased Retrotransposon Activity**: Deletion of these KRAB-ZFP gene clusters led to a significant        │
│ increase in the activity of the targeted retrotransposons (MusD and ETn). This indicates that these KRAB-ZFPs   │
│ play a crucial role in repressing the expression and mobilization of these retrotransposons.                    │
│                                                                                                                 │
│ 2. **Genomic Instability**: The increased activity of retrotransposons following the                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Now, we can compare the response when the model is prompted with the indexed PMC article as supporting context:

In [82]:
from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters

filters = MetadataFilters(
    filters=[
        ExactMatchFilter(key="file_name", value=FOCUS_ARTICLE.name),
    ]
)

query_engine = index.as_query_engine(
    llm=GEN_MODEL, filters=filters, similarity_top_k=3
)
result = query_engine.query(query)

console.print(
    Panel(
        result.response.strip(),
        title="Generated Content with RAG",
        border_style="bold green",
    )
)

I0923 13:41:58.302484   96846 chttp2_transport.cc:1402] ipv4:127.0.0.1:41175: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0923 13:41:58.302660   96846 chttp2_transport.cc:1434] ipv4:127.0.0.1:41175: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
2026-09-23 13:42:07,415 - INFO - HTTP Request: POST https://router.huggingface.co/v1/chat/completions "HTTP/1.1 200 OK"


╭────────────────────────────────────────── Generated Content with RAG ───────────────────────────────────────────╮
│ In this study, murine KRAB-ZFPs targeted several retrotransposon families, including ETn, IAP, and L1. The      │
│ CRISPR/Cas9 deletion of KRAB-ZFP gene clusters revealed that the deletion led to the reactivation of these      │
│ retrotransposons. Specifically, the deletion of the Chr2-cl cluster resulted in the reactivation of L1          │
│ subfamilies, RLTR10, and IAPEz ERVs, while the deletion of the Chr4-cl cluster led to the reactivation of       │
│ ETn/ETnERV elements. Additionally, the deletion of the Chr13.2-cl cluster also resulted in the upregulation of  │
│ ETn/ETnERV elements, and the deletion of the Chr13.1-cl cluster led to the upregulation of MMERVK10C elements.  │
│ These findings indicate that the KRAB-ZFPs play a crucial role in suppressing the activity of these             │
│ retrotransposons.                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

I0923 13:43:18.416641   96845 chttp2_transport.cc:1402] ipv4:127.0.0.1:41175: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0923 13:43:18.416694   96845 chttp2_transport.cc:1434] ipv4:127.0.0.1:41175: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
